In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

KeyboardInterrupt: 

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [ ]:
v1.shape

(384,)

In [ ]:
v2 = model.encode(q2)

In [ ]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)

np.float32(0.32332397)

In [ ]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [ ]:
v2.dot(dv)

np.float32(0.019730438)

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-27 04:28:35--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.3’

ingest.py.3         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-27 04:28:35 (35.4 MB/s) - ‘ingest.py.3’ saved [888/888]



In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents[4]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: What can I do before the course starts?',
 'answer': "Get the basic environment ready ahead of time:\n\n- Google Cloud account (free trial — see the GCP setup FAQ).\n- Google Cloud SDK (`gcloud` CLI).\n- Python 3 — install via your OS package manager or [`uv`](https://docs.astral.sh/uv/) (recommended for managing Python versions and project venvs).\n- Terraform.\n- Git.\n\nThen look over the prerequisites and syllabus to see if you're comfortable with the topics.",
 'doc_id': '33fc260cd8'}

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
texts[0], len(texts)

("Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 1350)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [ ]:
vectors[2]

array([-8.89654905e-02, -6.12818450e-02,  7.75604835e-03, -1.48411905e-02,
        5.76541945e-02,  3.91595028e-02, -1.01738960e-01,  5.06813452e-03,
       -1.13026373e-01,  8.81779846e-03,  2.82028448e-02,  2.14396417e-02,
       -4.82825236e-03,  4.61629517e-02,  3.01450063e-02,  3.44363004e-02,
       -4.42704968e-02, -5.48175983e-02,  5.62665351e-02, -1.81908570e-02,
       -2.86035463e-02, -3.57326642e-02, -3.72983702e-02,  3.58115509e-02,
        6.05605505e-02,  1.14685394e-01,  7.00178817e-02,  2.85895821e-03,
        3.82801332e-02,  6.65668072e-03, -6.63071051e-02,  1.49549926e-02,
        1.08069582e-02, -2.49716863e-02,  9.01510268e-02, -1.19031873e-02,
        5.90765141e-02, -2.70682890e-02, -2.43295468e-02, -2.43635513e-02,
       -4.54951115e-02, -4.48936187e-02, -6.00351766e-02,  6.58893734e-02,
       -3.77621246e-03,  1.72289424e-02,  2.23840680e-02, -9.53664258e-02,
       -5.73816635e-02,  3.99230272e-02,  6.39053583e-02,  2.78499536e-03,
       -1.00702748e-01, -

In [ ]:
import numpy as np
X = np.array(vectors)

In [ ]:
scores = X.dot(v1)

In [ ]:
scores 

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [ ]:
q1

'Can I still join the course after the start date?'

In [ ]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

In [ ]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [ ]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [ ]:
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [ ]:
np.argsort(scores)[-5:]

array([  7, 538, 907, 625,   2])

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
print(vindex)
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[4]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I get support if I take the course in the self-paced mode?',
 'answer': 'Yes, the Slack channel remains open and you can ask questions there. However, always search the channel first and check the FAQ, as most likely your questions are already answered here.\n\nYou can also tag the bot `@ZoomcampQABot` to help you conduct the search, but don’t rely on its answers 100%.',
 'doc_id': 'c207b8614e'}

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [ ]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


--2026-06-27 04:30:21--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-27 04:30:21 (34.6 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [ ]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes — you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
vs_index

In [ ]:
len(vectors)

1350

In [ ]:
len(documents)

1350

In [ ]:
vs_index.fit(vectors, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
len(query_vector)

384

In [ ]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [ ]:
vs_index.close()

NameError: name 'vs_index' is not defined

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
 keyword_fields=["course"],
 mode="ivf",
 db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [4]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

In [5]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
 keyword_fields=["course"],
 mode="ivf",
 db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [7]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join.\n\nYou don’t need a confirmation email, and you can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [8]:
vs_index.close()